# Chameleon 早期融合Token-Only多模态模型

目前为止我们见到的所有VLM都把图像和文本分离。视觉token从视觉编码器来，流过投影层，然后在LLM中碰到文本。视觉和文本的词表永远不重叠。Chameleon问，如果重合呢？训练一个VQ-VAE将一张图片转换成共享词表中离散token的序列。每个多模态文档都是文本token和图像token交错的序列，单个自回归损失。副作用是，模型可以产生混合模态输出，一次推理中交替产生文本和图像token。

## 问题描述

基于适配的VLM将文本和图片当做两种不同的东西，文本token流过`embed(text_token)`，图像流过`visual_encoder(image)-> projector -> pseudo_tokens`。模型有两种输入路径，然后在中途融合。

导致的三个结果。
- LLM只能消费图片，不能吐出图片。输出只有文本
- 多模态的文档（文本图片交叉出现）很尴尬，你只能在模型外解析多模态输入
- 分布不匹配。 视觉token和文本token住在隐空间的不同区域，带来的对齐问题。

Chameleon 拒绝这种前提：图片只是从共享词表中获取的离散token序列。在交叉文档上训练，一个损失，一个自回归解码器，你就免费解锁了多模态生成。

## 基本概念

### VQ-VAE 当做图片分词器

分词器是一个矢量量化变分自编码器（vector-quantized variational autoencoder）。架构：
- 编码器。 CNN + ViT 将图片映射到空间特征图，32x32特征即256维。
- 码本。 一个可学习的词表，包含K个256维向量（Chameleon 中8192）
- 量化。 对于每个空间特征，在码本中找到L2距离最近的编码。将连续特征替换成整数索引。
- 解码器。 CNN 输入量化特征，生成像素。

训练：VAE重建损失（解码出来的图要像原图） + 码本损失（向量要符合生产中实际分布）+ 承诺损失（解码的向量不要离码本向量太远）。人话就是编得好、码本有用、编码器愿意贴码本。码本索引构成了图片的字符集。

对于Chameleon来说，一张图片变成了32x32=1024个token，需要从大小为8192的码本中来。将其连接到文本token，比如32000大小的BPE词表，最终的词表的大小为40192。Transformer只看一个序列，计算一个损失。

### 共享词表

Chameleon的词表包含文本token，图片token以及模态分隔符。每个token都有单独的ID。输入嵌入层将每个ID映射到一个D维的隐向量。输出投影将隐向量投影回选词概率，不管模态是什么。

分隔符很重要：`<image>`和`</image>`标签将图片token序列包裹。在生成的时候，如果模型吐出`<image>`，下游的软件就知道接下来1024个词元是VQ 索引，并送到解码器渲染像素。

### 混合模态生成

推理是在共享词表中预测下一个token。比如提示词“画一张猫然后描述它”。Chameleon吐出：
```
<image> 123 2135 435 ... (1024个图片token) </image>
橘猫，坐在窗沿。
```
模型自己挑输出顺序，图片和文本顺序不固定。

相对于适配器VLM只能生产文本，Chameleon唤起了模型输出多模态内容的问题。

### 训练稳定性————QK-Norm， dropout，LayerNorm顺序

早期融合训练非常不稳定，Chameleon给了三个技巧
- QK-Norm。 在注意力中对Q 和 K 应用层归一化，防止logit 爆炸。
- Dropout。  在每次残差连接后dropout，而不是只有注意力和MLP后。来自图像token的梯度占主导时，需要更多的归一化。
- LayerNorm顺序。 残差分支用Pre-LN（标准），额外对跳连加上一个LN。稳定末层梯度流。

### Chameleon 对比BLIP-2/LLaVA

Chameleon走早期融合，共享词表：
- 一个损失，一个解码器
- 可以生产多模态输出
- 分词器是质量天花板
- 贵。推理的时候生产图像走VQ-VAE解码器

BLIP-2/LLaVA 走后融合：
- 视觉输入，但输出只有文本
- 需要预训练好的LLM
- 理解上没有分词器的瓶颈
- 便宜。推理的时候一次前向就可以

根据任务类型选择，如果需要图片生成，选Chameleon族，如果只需要理解，适配器VLM更简单，而且节约更多预训练成本。

### Fuyu 和 AnyGPT

Fuyu 是一个相近的尝试。完全跳过单独的视觉编码器，将原生的图片patch直接通过线性层当做LLM的输入。比Chameleon简单，但是失去了共享词表的输出生成能力。

AnyGPT拓展Chanmeleon到四个模态：文本、图片、演讲、音乐。相同的VQ-VAE技巧。后续的Any-to-any 覆盖了更多。


# 开始编码

教学积木：Chameleon 核心——**VQ-VAE 图像分词**、**共享词表早融合**、**QK-Norm + 残差 dropout / 额外 LN**、**混合模态生成示意**。

> 说明：磁盘上原 `code.ipynb` 为空；上方概念笔记按编辑器内容一并写入，请核对是否与你本地一致。


## 1. 玩具 VQ-VAE：连续特征 → 码本索引 → 重建


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyChameleonConfig:
    """Chameleon 教学配置（远小于真模型）。"""

    image_size: int = 32
    """输入方图边长。"""

    latent_side: int = 4
    """量化特征图边长；真 Chameleon 约 32 → 1024 tokens。"""

    codebook_size: int = 64
    """图像码本大小 K（真模型 8192）。"""

    codebook_dim: int = 32
    """码本向量维 / 量化特征维。"""

    text_vocab_size: int = 128
    """纯文本 BPE 词表大小示意。"""

    dim: int = 64
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1
    beta_commit: float = 0.25
    """承诺损失系数。"""

    @property
    def num_image_tokens(self) -> int:
        """一张图展开后的 VQ 索引个数。"""
        return self.latent_side * self.latent_side

    @property
    def img_token_offset(self) -> int:
        """图像码本 id 在共享词表中的起始偏移（紧接文本词表之后）。"""
        return self.text_vocab_size

    @property
    def shared_vocab_size(self) -> int:
        """文本 + 图像码本 + 4 个特殊符。"""
        return self.text_vocab_size + self.codebook_size + 4

    @property
    def id_bos(self) -> int:
        return self.text_vocab_size + self.codebook_size

    @property
    def id_eos(self) -> int:
        return self.id_bos + 1

    @property
    def id_img_start(self) -> int:
        """``<image>``。"""
        return self.id_bos + 2

    @property
    def id_img_end(self) -> int:
        """``</image>``。"""
        return self.id_bos + 3


class VectorQuantizer(nn.Module):
    """矢量量化：每个空间向量替换为码本中 L2 最近邻的索引（STE 反传）。"""

    def __init__(self, codebook_size: int, dim: int) -> None:
        super().__init__()
        self.codebook = nn.Embedding(codebook_size, dim)
        nn.init.uniform_(self.codebook.weight, -1.0 / codebook_size, 1.0 / codebook_size)

    def quantize(
        self,
        z_e: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            z_e: ``(B, D, H, W)`` 编码器连续特征。

        Returns:
            z_q: ``(B, D, H, W)`` STE 量化特征。
            indices: ``(B, H*W)`` 码本整数索引。
            codebook_loss: 码本贴近 encoder 输出。
            commit_loss: encoder 贴近码本。
        """
        B, D, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)
        x2 = (flat**2).sum(dim=1, keepdim=True)
        e2 = (self.codebook.weight**2).sum(dim=1)
        dist = x2 + e2.unsqueeze(0) - 2.0 * flat @ self.codebook.weight.t()
        idx = dist.argmin(dim=1)
        z_q_flat = self.codebook(idx)
        codebook_loss = F.mse_loss(z_q_flat.detach(), flat)
        commit_loss = F.mse_loss(z_q_flat, flat.detach())
        z_q_flat = flat + (z_q_flat - flat).detach()
        z_q = z_q_flat.view(B, H, W, D).permute(0, 3, 1, 2).contiguous()
        indices = idx.view(B, H * W)
        return z_q, indices, codebook_loss, commit_loss


class TinyVQVAE(nn.Module):
    """玩具 VQ-VAE：浅 CNN 编解码 + 码本量化。"""

    def __init__(self, cfg: TinyChameleonConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, cfg.codebook_dim, 4, 2, 1),
        )
        self.quant = VectorQuantizer(cfg.codebook_size, cfg.codebook_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(cfg.codebook_dim, 64, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh(),
        )

    def encode_to_indices(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: ``(B, 3, H, W)``，``H=W=image_size``。

        Returns:
            indices: ``(B, latent_side*latent_side)``。
        """
        z_e = self.encoder(images)
        _, indices, _, _ = self.quant.quantize(z_e)
        return indices

    def decode_from_indices(self, indices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            indices: ``(B, N)``，``N=latent_side**2``。

        Returns:
            images: ``(B, 3, H, W)``。
        """
        B, N = indices.shape
        S = self.cfg.latent_side
        if N != S * S:
            raise ValueError(f"expect {S*S} indices, got {N}")
        z_q = self.quant.codebook(indices).view(B, S, S, -1).permute(0, 3, 1, 2)
        return self.decoder(z_q)

    def forward(
        self,
        images: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            images: ``(B, 3, H, W)``。

        Returns:
            recon: 重建图。
            indices: VQ 索引。
            loss: ``recon + codebook + beta * commit``。
        """
        z_e = self.encoder(images)
        z_q, indices, codebook_loss, commit_loss = self.quant.quantize(z_e)
        recon = self.decoder(z_q)
        recon_loss = F.mse_loss(recon, images)
        loss = recon_loss + codebook_loss + self.cfg.beta_commit * commit_loss
        return recon, indices, loss


print("TinyVQVAE ready")


## 2. 共享词表早融合 Transformer（QK-Norm、残差 dropout、跳连 LN）


In [ ]:
class QKNormAttention(nn.Module):
    """注意力前对 Q、K 做 LayerNorm，抑制早期融合时的 logit 爆炸。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        if dim % n_heads:
            raise ValueError("dim must divide n_heads")
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)
        self.q_norm = nn.LayerNorm(self.head_dim)
        self.k_norm = nn.LayerNorm(self.head_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。

        Returns:
            y: ``(B, L, D)`` 因果自注意力输出（无残差）。
        """
        B, L, D = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = self.q_norm(q).transpose(1, 2)
        k = self.k_norm(k).transpose(1, 2)
        v = v.transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        causal = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device, dtype=x.dtype), 1
        )
        attn = (attn + causal).softmax(dim=-1)
        attn = self.drop(attn)
        h = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out(h)


class ChameleonBlock(nn.Module):
    """Pre-LN 主路径 + 残差后 dropout；并对跳连再加 LN。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = QKNormAttention(dim, n_heads, dropout)
        self.drop1 = nn.Dropout(dropout)
        self.skip_ln1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )
        self.drop2 = nn.Dropout(dropout)
        self.skip_ln2 = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。

        Returns:
            y: ``(B, L, D)``。
        """
        a = self.attn(self.norm1(x))
        x = self.skip_ln1(x + self.drop1(a))
        m = self.mlp(self.norm2(x))
        x = self.skip_ln2(x + self.drop2(m))
        return x


class EarlyFusionAR(nn.Module):
    """单路自回归 Transformer：共享嵌入 / 共享输出头。"""

    def __init__(self, cfg: TinyChameleonConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.tok = nn.Embedding(cfg.shared_vocab_size, cfg.dim)
        self.blocks = nn.ModuleList(
            [ChameleonBlock(cfg.dim, cfg.n_heads, cfg.dropout) for _ in range(cfg.n_layers)]
        )
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.shared_vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: ``(B, L)`` 共享词表 id。

        Returns:
            logits: ``(B, L, shared_vocab_size)``。
        """
        x = self.tok(input_ids)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x))

    def next_token_loss(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            input_ids: ``(B, L)``。

        Returns:
            loss: 因果 LM 交叉熵。
        """
        logits = self.forward(input_ids[:, :-1])
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            input_ids[:, 1:].reshape(-1),
        )


def image_indices_to_shared_ids(indices: torch.Tensor, cfg: TinyChameleonConfig) -> torch.Tensor:
    """
    Args:
        indices: ``(B, N)`` 码本下标 ``[0, K)``。
        cfg: 配置。

    Returns:
        ids: ``(B, N)`` 共享词表图像区间 id。
    """
    return indices + cfg.img_token_offset


def shared_ids_to_image_indices(ids: torch.Tensor, cfg: TinyChameleonConfig) -> torch.Tensor:
    """
    Args:
        ids: ``(B, N)`` 共享词表图像 token。
        cfg: 配置。

    Returns:
        indices: ``(B, N)`` 码本下标。
    """
    return ids - cfg.img_token_offset


def build_multimodal_sequence(
    text_prefix: torch.Tensor,
    image_indices: torch.Tensor,
    text_suffix: torch.Tensor,
    cfg: TinyChameleonConfig,
) -> torch.Tensor:
    """
    构造 ``prefix + <image> + VQ tokens + </image> + suffix``。

    Args:
        text_prefix: ``(B, Lp)``。
        image_indices: ``(B, N)`` 原始 VQ 索引。
        text_suffix: ``(B, Ls)``。
        cfg: 配置。

    Returns:
        seq: ``(B, Lp+1+N+1+Ls)``。
    """
    B = text_prefix.size(0)
    start = torch.full((B, 1), cfg.id_img_start, device=text_prefix.device, dtype=torch.long)
    end = torch.full((B, 1), cfg.id_img_end, device=text_prefix.device, dtype=torch.long)
    img_ids = image_indices_to_shared_ids(image_indices, cfg)
    return torch.cat([text_prefix, start, img_ids, end, text_suffix], dim=1)


print("EarlyFusionAR + shared vocab helpers ready")


## 3. 混合模态生成示意：``<image>`` 后收集 VQ token 并解码


In [ ]:
@torch.no_grad()
def greedy_generate_mixed(
    model: EarlyFusionAR,
    vqvae: TinyVQVAE,
    prompt_ids: torch.Tensor,
    max_new_tokens: int = 32,
) -> tuple[torch.Tensor, torch.Tensor | None]:
    """
    贪心解码共享词表；若出现 ``<image>``，再读满图像 token 并用 VQ 解码。

    Args:
        model: 早融合 AR。
        vqvae: 图像分词器/解码器。
        prompt_ids: ``(1, L0)``。
        max_new_tokens: 最多新生成 token 数。

    Returns:
        full_ids: ``(1, L)``。
        rendered: ``(1, 3, H, W)`` 或 ``None``。
    """
    cfg = model.cfg
    ids = prompt_ids.clone()
    collecting = False
    img_buf: list[int] = []
    rendered: torch.Tensor | None = None

    for _ in range(max_new_tokens):
        logits = model(ids)[:, -1, :]
        next_id = int(logits.argmax(dim=-1).item())
        ids = torch.cat([ids, torch.tensor([[next_id]], device=ids.device)], dim=1)

        if not collecting and next_id == cfg.id_img_start:
            collecting = True
            img_buf = []
            continue
        if collecting:
            if next_id == cfg.id_img_end:
                if len(img_buf) == cfg.num_image_tokens:
                    idx = torch.tensor([img_buf], device=ids.device, dtype=torch.long)
                    idx = shared_ids_to_image_indices(idx, cfg)
                    rendered = vqvae.decode_from_indices(idx)
                collecting = False
                img_buf = []
            elif cfg.img_token_offset <= next_id < cfg.img_token_offset + cfg.codebook_size:
                img_buf.append(next_id)
                if len(img_buf) > cfg.num_image_tokens:
                    collecting = False
                    img_buf = []
        if next_id == cfg.id_eos:
            break
    return ids, rendered


print("mixed generation helper ready")


## 4. 冒烟测试


In [ ]:
def smoke_test() -> None:
    """验证 Chameleon 教学要点。"""
    torch.manual_seed(0)
    cfg = TinyChameleonConfig()
    vq = TinyVQVAE(cfg)
    ar = EarlyFusionAR(cfg)

    print("=== VQ-VAE ===")
    images = torch.randn(2, 3, cfg.image_size, cfg.image_size)
    recon, indices, vq_loss = vq(images)
    print(f"recon={tuple(recon.shape)}, indices={tuple(indices.shape)}, loss={vq_loss.item():.4f}")
    assert indices.shape == (2, cfg.num_image_tokens)

    print("\n=== shared vocab early fusion LM ===")
    prefix = torch.randint(0, cfg.text_vocab_size, (2, 3))
    suffix = torch.randint(0, cfg.text_vocab_size, (2, 2))
    seq = build_multimodal_sequence(prefix, indices, suffix, cfg)
    print(f"seq={tuple(seq.shape)} vocab={cfg.shared_vocab_size}")
    assert (seq[:, 3] == cfg.id_img_start).all()
    loss = ar.next_token_loss(seq)
    loss.backward()
    print(f"AR loss={loss.item():.4f}")

    print("\n=== decode indices ===")
    pix = vq.decode_from_indices(indices)
    print(f"decoded={tuple(pix.shape)}")

    print("\n=== parse <image> span ===")
    prompt = torch.tensor([[cfg.id_bos, 7, 8]], dtype=torch.long)
    fake_img = image_indices_to_shared_ids(indices[:1], cfg)
    forced = torch.cat(
        [
            prompt,
            torch.tensor([[cfg.id_img_start]], dtype=torch.long),
            fake_img,
            torch.tensor([[cfg.id_img_end, cfg.id_eos]], dtype=torch.long),
        ],
        dim=1,
    )
    start = int((forced[0] == cfg.id_img_start).nonzero()[0])
    end = int((forced[0] == cfg.id_img_end).nonzero()[0])
    span = forced[:, start + 1 : end]
    rend = vq.decode_from_indices(shared_ids_to_image_indices(span, cfg))
    print(f"forced render={tuple(rend.shape)}")
    print("SMOKE TEST OK")


smoke_test()
